In [5]:
# REPRODUCIBILITY SETUP CELL

from pathlib import Path
import os
import numpy as np
import random

print("Setting up reproducible environment...\n")


# Automatically locate project root

current_dir = Path.cwd()

if current_dir.name == "sentiment-analysis-tool":
    PROJECT_ROOT = current_dir

elif (current_dir / "sentiment-analysis-tool").exists():
    PROJECT_ROOT = current_dir / "sentiment-analysis-tool"
    os.chdir(PROJECT_ROOT)

elif (current_dir.parent / "sentiment-analysis-tool").exists():
    PROJECT_ROOT = current_dir.parent / "sentiment-analysis-tool"
    os.chdir(PROJECT_ROOT)

else:
    raise Exception(
        "Project folder 'sentiment-analysis-tool' not found.\n"
        "Make sure the repository is cloned correctly."
    )

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())



# Define Standard Project Paths

DATA_RAW = PROJECT_ROOT / "data" / "raw_data"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_ANNOTATED = PROJECT_ROOT / "data" / "annotated"

MODELS = PROJECT_ROOT / "models"
NOTEBOOKS = PROJECT_ROOT / "notebooks"

print("\nProject paths ready.")



# Create Missing Folders Automatically

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_ANNOTATED.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print("Required folders verified.")



# Fix Random Seeds for Reproducibility

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

print("\nRandom seed set to:", SEED)


# Display Python Environment

import sys

print("\nPython version:", sys.version.split()[0])


print("\nReproducibility setup complete.")

Setting up reproducible environment...

Project root: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool
Working directory: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool

Project paths ready.
Required folders verified.

Random seed set to: 42

Python version: 3.11.0

Reproducibility setup complete.


In [6]:
# Data annotation and feature engineering

# Importing necessary libraries
import pandas as pd
import numpy as np
import re
import os

from pathlib import Path

# Machine Learning Imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.decomposition import NMF

# Reproducibility
RANDOM_STATE = 42

# Use current folder as project root
PROJECT_ROOT = Path.cwd()

print("Working directory:", PROJECT_ROOT)

# Load dataset
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_final_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

df.head()

Working directory: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool
Dataset shape: (52943, 6)


,source,date,channel,product,text,language
0,Apple Store,06/02/2026 00:00,app store reviews,android mobile banking app,account cant allow me to send money to someone...,english
1,Apple Store,30/01/2026 00:00,app store reviews,android mobile banking app,i wanted to send money to another bank only to...,english
2,Apple Store,27/01/2026 00:00,app store reviews,android mobile banking app,finishing to launch only logo appears,unknown
3,Apple Store,27/01/2026 00:00,app store reviews,android mobile banking app,isnt working since i last updated,unknown
4,Apple Store,23/01/2026 00:00,app store reviews,android mobile banking app,quick and reliable,english


In [7]:
df['text'].isna().sum()


np.int64(20)

In [8]:
# Drop rows with missing values in the 'text' column
df = df.dropna(subset=['text'])


In [9]:
# Remove rows where 'text' is empty or contains only whitespace
df = df[df['text'].str.strip() != ""]


In [10]:
# Check for null values and empty strings again
print("Null values:", df['text'].isna().sum())
print("Empty strings:", (df['text'].str.strip() == "").sum())


Null values: 0
Empty strings: 0


In [11]:
# Text cleaning for topic modeling

def clean_text(text):

    text = str(text).lower()

    # Remove HTML artifacts
    text = re.sub(r'nbsp\w*', ' ', text)

    # Remove emails
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove repeated words
    text = re.sub(r'\b(\w+)( \1\b)+', r'\1', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


df['clean_text'] = df['text'].apply(clean_text)

print("Cleaning complete")

Cleaning complete


In [12]:
# Topic modeling option 1: LDA

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    max_df=0.8,
    min_df=3,
    stop_words=None
)

X = vectorizer.fit_transform(df['clean_text'])

lda = LatentDirichletAllocation(
    n_components=5,
    random_state=RANDOM_STATE
)

lda.fit(X)

print("LDA completed")

LDA completed


In [13]:
print("TF-IDF shape:", X.shape)


TF-IDF shape: (52923, 5000)


In [14]:
# Get feature names and top words for each topic
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(lda.components_):
    top_features_idx = topic.argsort()[::-1][:10]  # descending order
    top_features = [feature_names[i] for i in top_features_idx]
    
    print(f"\nTopic {topic_idx + 1}:")
    print(top_features)



Topic 1:
['assist', 'card', 'attached', 'the', 'accountnumber', 'to', 'for', 'assist to', 'cover', 'as']

Topic 2:
['my', 'to', 'app', 'account', 'password', 'me', 'the', 'in', 'help', 'is']

Topic 3:
['the', 'of', 'you', 'for', 'to', 'and', 'your', 'confirm', 'this', 'is']

Topic 4:
['accountnumber', 'to', 'phone', 'the', 'name', 'number', 'assist', 'month', 'one month', 'wrong']

Topic 5:
['email', 'the', 'and', 'or', 'of', 'this', 'you', 'any', 'is', 'by']


In [15]:
df.head()

,source,date,channel,product,text,language,clean_text
0,Apple Store,06/02/2026 00:00,app store reviews,android mobile banking app,account cant allow me to send money to someone...,english,account cant allow me to send money to someone...
1,Apple Store,30/01/2026 00:00,app store reviews,android mobile banking app,i wanted to send money to another bank only to...,english,i wanted to send money to another bank only to...
2,Apple Store,27/01/2026 00:00,app store reviews,android mobile banking app,finishing to launch only logo appears,unknown,finishing to launch only logo appears
3,Apple Store,27/01/2026 00:00,app store reviews,android mobile banking app,isnt working since i last updated,unknown,isnt working since i last updated
4,Apple Store,23/01/2026 00:00,app store reviews,android mobile banking app,quick and reliable,english,quick and reliable


In [16]:
# Topic modeling option 2: NMF (Better for short text)

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    max_df=0.75,
    min_df=3,
    stop_words=None
)

X_tfidf = tfidf.fit_transform(df['clean_text'])

nmf = NMF(
    n_components=5,
    random_state=RANDOM_STATE
)

W = nmf.fit_transform(X_tfidf)
H = nmf.components_

print("NMF completed")

NMF completed


In [17]:
# Get feature names and top words for each topic
feature_names = tfidf.get_feature_names_out()

n_top_words = 10

for topic_idx, topic in enumerate(nmf.components_):
    print(f"\nTopic {topic_idx + 1}:")
    top_features = topic.argsort()[-n_top_words:][::-1]
    print([feature_names[i] for i in top_features])


Topic 1:
['the', 'and', 'of', 'this', 'you', 'is', 'to', 'email', 'or', 'for']

Topic 2:
['my', 'app', 'password', 'to', 'me', 'help', 'mobile', 'my password', 'my account', 'account']

Topic 3:
['accountnumber', 'assist', 'number', 'card', 'to', 'phone', 'assist to', 'no', 'idnumber', 'id']

Topic 4:
['email', 'name', 'email wrote', 'wrote', 'mpesa', 'you', 'pm', 'name name', 'name email', 'you for']

Topic 5:
['cover', 'issue', 'month', 'one month', 'one', 'issue one', 'annual', 'month cover', 'certificate', 'attached']


### Attempt 1: LDA Topic Modelling
Results were not sufficiently interpretable.

### Attempt 2: NMF Topic Modelling
Tested as an alternative approach better suited for short texts.


Topic Mapping
Topic 1 = Account Verification (account number, id number, card, assist)
Topic 2 = Loans and Services (loan, account, idnumber)
Topic 3 = General Inquiry (the, you, email)
Topic 4 = Insurance Services (cover, certificate, annual)
Topic 5 = Mobile Banking App (app, password, mobile)

Opted for manual labelling since the topis have weak keywords

In [50]:
# Create annotation file with topic labels

df_annotation = df.copy()

df_annotation['topic'] = ""
df_annotation['sentiment'] = ""
df_annotation['annotator'] = ""
df_annotation['confidence'] = ""

In [54]:
# Save annotation file

SAVE_PATH = PROJECT_ROOT / "data" / "annotated_data"

SAVE_PATH.mkdir(exist_ok=True)

df_annotation.to_excel(
    SAVE_PATH / "annotation_file.xlsx",
    index=False
)

print("Annotation file saved")

Annotation file saved


In [55]:
# sample 2000 rows for annotation
sample_df = df.sample(2000, random_state=42)

sample_df.to_csv(
    PROJECT_ROOT / "data" / "annotated_data" / "sample_dataset_for_annotation.csv",
    index=False
)

Sampling was not all inclusing. Language: Only English and Sheng rows were picked. Swahili rows were ommitted. The sampled sata was not of good quality. Manual sampling was done followed by a manual annotation process



Next Step: Manual annotation process and column merges to match original dataset columns


In [58]:
# Load manually annotated merged dataset

ANNOTATED_PATH = PROJECT_ROOT / "data" / "annotated_data" / "final_annotated_dataset.xlsx"

df_annotated = pd.read_excel(ANNOTATED_PATH)

print(df_annotated.shape)

df_annotated.head()

(2000, 14)


,text,topic,sentiment,annotator,confidence,match_key,match_text,source,date,channel,product,language,clean_text,match_score
0,i am writing to seek urgent assistance regardi...,Mobile Banking App,negative,annotator_1,high,i am writing to seek urgent assistance regardi...,i am writing to seek urgent assistance regardi...,CRM Tool,04/11/2025 00:00,Service Email,Mobile,english,i am writing to seek urgent assistance regardi...,99.737877
1,name seem to neglect your duty to give the bes...,Mobile Banking App,negative,annotator_1,high,name seem to neglect your duty to give the bes...,name seem to neglect your duty to give the bes...,CRM Tool,05/11/2025 00:00,Service Email,Mobile,english,seem to neglect your duty to give the best bec...,98.245614
2,namere seeking your assistance in providing ...,Payments,negative,annotator_1,high,namere seeking your assistance in providing a ...,namere seeking your assistance in providing a ...,CRM Tool,05/12/2025 00:00,Service Email,Thunes,english,namere seeking your assistance in providing a ...,88.442623
3,reminder on resolving this issue of payment ...,Acccount Issues,negative,annotator_1,high,reminder on resolving this issue of payment on...,reminder on resolving this issue of payment on...,CRM Tool,19/11/2025 00:00,Service Email,Card Online Disputes,english,name name reminder on resolving this issue of ...,97.071130
4,a member of you service and am trying to open...,Mobile Banking App,negative,annotator_1,high,a member of you service and am trying to open ...,a member of you service and am trying to open ...,CRM Tool,25/11/2025 00:00,Service Email,Mobile,english,namename a member of you service and am trying...,97.352342


In [59]:
# Check for missing values
df_annotated[['topic','sentiment','confidence']].isnull().sum()

topic         0
sentiment     0
confidence    0
dtype: int64

In [60]:
# Check for empty labels
df_annotated['topic'].value_counts()


topic
Mobile Banking App    798
Insurance Services    363
Loans                 306
Customer Service      157
Acccount Issues       152
Payments              126
Cards                  72
General Inquiry        16
ATM Machines           10
Name: count, dtype: int64

In [61]:
# Check for empty labels
df_annotated['sentiment'].value_counts()

sentiment
negative    1003
neutral      806
positive     191
Name: count, dtype: int64

In [62]:

# Check confidence distribution
df_annotated['confidence'].value_counts()

confidence
high      1839
medium     158
low          3
Name: count, dtype: int64

In [63]:
# Check annotator distribution
df_annotated['annotator'].value_counts()

annotator
annotator_1    2000
Name: count, dtype: int64

In [65]:
# Drop rows with missing values in key annotation columns
df_annotated = df_annotated.dropna(
    subset=['text','topic','sentiment']
)

In [66]:
# Save final annotated dataset
FINAL_PATH = PROJECT_ROOT / "data" / "annotated_data"

df_annotated.to_csv(
    FINAL_PATH / "final_annotated_dataset.csv",
    index=False
)

print("Final dataset saved")

Final dataset saved


In [67]:
df_annotated.shape

(2000, 14)